# Multi-Step Layer Merging / Removal — GPT-2 / Kaggle

This notebook compares **multiple rounds of layer compression (1–5 steps)** for GPT-2 small.

The important point is that, from the second merge onward, merging is performed **with respect to the current group structure**. Therefore, the following two cases are treated as different candidates.

- Non-overlapping: merge `1-2`, then merge a separate pair `3-4` → `[1,2]` and `[3,4]`
- Chained: merge `1-2`, then merge the resulting merged block with `3` → `[1,2,3]`

For chained NonlinearMultiply merges, the previously merged block is used as the initialization, while the teacher target is the **full span of the original layers**. For example, with three layers,

\[
\Phi_3(\Phi_2(\Phi_1(x))) pprox \Phi_{merged}(x)
\]

is learned directly.

| Method | Definition for multiple layers |
|---|---|
| LinearMultiply | Approximate each original layer in the span with an affine transformation and compose them sequentially into a single affine block |
| Remove | Keep only the first block of the combined group and remove the others |
| Average | Take an **equal-weight average of the original blocks** in the combined group (e.g., 1/3 each for three layers) |
| NonlinearMultiply | Initialize from the current left merged block and train using the original teacher's full-span hidden target + logit KD |

For each method, at every compression step, all merge candidates among the currently adjacent groups are evaluated on validation data, and the best candidate is advanced to the next stage using a **greedy stage-wise search**. Because this is not an exhaustive search over all combinations, the global optimum is not guaranteed, but candidates such as `1-2;3-4` and `1-3` can be compared at the same step.

Final evaluation is performed on **WikiText-2 test / LAMBADA / C4**, and none of these datasets are used for training or validation selection.


## 1. Kaggle Environment

- Accelerator: **GPU**
- Internet: **On**
- Run the cells from top to bottom.

With `full` + `max_merges=5`, the computational cost is substantially higher than in the original single-merge experiment because even NonlinearMultiply evaluates all candidates at every stage. It is recommended to first verify that everything works in `quick` mode and then run `full`.


In [1]:
%pip install -q --disable-pip-version-check transformers==4.57.6 datasets==4.4.1 scipy==1.16.3 pandas==2.2.3 threadpoolctl==3.6.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 82.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 90.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.

## 2. Set Up the Implementation

`affine_core.py` and `evaluation_protocol.py` are used unchanged from the original notebook, and `nonlinear_composition_multi.py` is added for the multi-step experiment.


In [2]:
%%writefile /kaggle/working/affine_core.py
"""Unchanged affine/evaluation utilities extracted from the supplied notebook.
Only functions needed by Nonlinear Composition Merging are included.
"""
from __future__ import annotations
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
import math
import random
import time
import numpy as np
import scipy
from scipy.linalg import cho_factor, cho_solve
import torch
from torch import nn
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


def _array(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    a = np.asarray(x, dtype=np.float64)
    if a.ndim != 2 or not np.isfinite(a).all():
        raise ValueError("Expected a finite [tokens, features] matrix")
    return a


def _relative_reg(gram, strength):
    if strength <= 0:
        raise ValueError("A positive ridge is required")
    scale = float(np.trace(gram) / gram.shape[0])
    return strength * max(scale, np.finfo(np.float64).eps)


def fit_affine(X, Y, ridge=1e-4):
    """min ||XA+b-Y||_F^2 + lambda ||A||_F^2; bias unpenalized.

    lambda = ridge * trace(Xc.T Xc)/d, matching the previous experiment.
    No explicit inverse; CPU float64 is intentional for stable CCA/ridge fits.
    """
    X, Y = _array(X), _array(Y)
    if len(X) != len(Y) or len(X) < 2:
        raise ValueError("X and Y need the same N >= 2")
    mx, my = X.mean(0), Y.mean(0)
    xc, yc = X - mx, Y - my
    gram = xc.T @ xc
    reg = _relative_reg(gram, ridge)
    A = cho_solve(cho_factor(gram + reg * np.eye(X.shape[1])), xc.T @ yc)
    b = my - mx @ A
    return A, b


class AffineMergedBlock(nn.Module):
    """GPT2Block ABI for transformers==4.57.6, no KV cache.

    This already approximates the WHOLE two-block map; do not add x again.
    """
    def __init__(self, A, b, device=None, dtype=torch.float32):
        super().__init__()
        d = A.shape[0]
        self.linear = nn.Linear(d, d, device=device, dtype=dtype)
        with torch.no_grad():
            self.linear.weight.copy_(torch.as_tensor(np.array(A.T, copy=True)))
            self.linear.bias.copy_(torch.as_tensor(np.array(b, copy=True)))

    def forward(self, hidden_states, past_key_values=None, cache_position=None,
                attention_mask=None, head_mask=None, encoder_hidden_states=None,
                encoder_attention_mask=None, use_cache=False, output_attentions=False,
                **kwargs):
        if use_cache or past_key_values is not None:
            raise ValueError("Affine merge supports use_cache=False and no past_key_values")
        if encoder_hidden_states is not None:
            raise ValueError("Cross-attention is outside this GPT-2 experiment")
        output = (self.linear(hidden_states),)
        return output + (None,) if output_attentions else output


def parameter_count(model):
    return sum(p.numel() for p in model.parameters())  # tied weights counted once


def make_state_windows(ids, total_tokens, chunk_size, seed=None):
    """Disjoint source windows; random order for TRAIN, sequential for VAL."""
    ids = ids.detach().cpu().flatten()
    if len(ids) < total_tokens or total_tokens < 2 or chunk_size < 2:
        raise ValueError("Invalid number of state tokens / chunk size")
    starts = np.arange(0, len(ids), chunk_size)
    if seed is not None:
        np.random.default_rng(seed).shuffle(starts)
    windows, remaining = [], total_tokens
    for start in starts:
        end = min(int(start) + chunk_size, len(ids), int(start) + remaining)
        if end <= start:
            continue
        windows.append(ids[int(start):end].clone())
        remaining -= end - int(start)
        if remaining == 0:
            break
    if remaining:
        raise ValueError("Insufficient tokens")
    return windows


@torch.inference_mode()
def collect_pair_states(model, windows, L):
    """Hooks capture pre-ln_f block outputs, including pair (10,11)."""
    if model.training:
        raise ValueError("Collect states with model.eval()")
    d = model.config.n_embd
    n = sum(w.numel() for w in windows)
    states = [np.empty((n, d), np.float32) for _ in range(3)]
    position = [0, 0, 0]
    def store(index, tensor):
        a = tensor.detach().reshape(-1, d).float().cpu().numpy()
        start = position[index]
        states[index][start:start + len(a)] = a
        position[index] += len(a)
    handles = [
        model.transformer.h[L].register_forward_pre_hook(lambda m, inp: store(0, inp[0])),
        model.transformer.h[L].register_forward_hook(lambda m, inp, out: store(1, out[0])),
        model.transformer.h[L + 1].register_forward_hook(lambda m, inp, out: store(2, out[0])),
    ]
    try:
        device = next(model.parameters()).device
        for window in windows:
            # No vocabulary logits needed while collecting states.
            model.transformer(window[None].to(device), use_cache=False, return_dict=True)
    finally:
        for handle in handles:
            handle.remove()
    if position != [n, n, n]:
        raise RuntimeError(f"Incomplete activation collection: {position}")
    return tuple(states)


@torch.inference_mode()
def measure_latency(model, input_ids, repeats=10, warmup=5):
    """Batch=1 full LM forward, no KV cache, excludes CPU->GPU transfer."""
    if repeats < 1:
        raise ValueError("Latency repeats must be positive")
    device = next(model.parameters()).device
    x = input_ids.flatten()[None].to(device)
    def sync():
        if device.type == "cuda":
            torch.cuda.synchronize(device)
    model.eval()
    for _ in range(warmup):
        model(x, use_cache=False)
    sync()
    samples = []
    for _ in range(repeats):
        start = time.perf_counter()
        model(x, use_cache=False)
        sync()
        samples.append(1000 * (time.perf_counter() - start))
    return {"Latency_ms": float(np.median(samples)),
            "Latency_Mean_ms": float(np.mean(samples)),
            "Latency_P90_ms": float(np.quantile(samples, 0.9))}


def check_transformers_version():
    if transformers.__version__ != "4.57.6":
        raise RuntimeError("Install transformers==4.57.6 and restart the Kaggle session before importing.")



Writing /kaggle/working/affine_core.py


In [3]:
%%writefile /kaggle/working/evaluation_protocol.py
"""Exact sliding/legacy evaluator from merging_comparison_colab.ipynb."""
import json
import math
from pathlib import Path
import numpy as np
import torch

def target_mask(length, context, protocol):
    """Global positions whose next-token losses belong in the denominator."""
    if context < 2 or length < 2:
        raise ValueError("Need context and token count >= 2")
    mask = torch.zeros(length, dtype=torch.bool)
    if protocol == "sliding":
        mask[1:] = True
    elif protocol == "legacy":
        for start in range(0, (length//context)*context, context):
            mask[start+1:start+context] = True
    else:
        raise ValueError("protocol must be sliding or legacy")
    return mask


def protocol_windows(ids, context, stride, protocol="sliding", score_mask=None):
    ids = ids.detach().cpu().flatten()
    targets = target_mask(len(ids), context, protocol)
    if score_mask is not None:
        if score_mask.shape != targets.shape:
            raise ValueError("score_mask must match the token sequence")
        targets &= score_mask.cpu().bool()
    if not targets.any():
        raise ValueError("No scored tokens; legacy requires a full context window")
    if protocol == "sliding":
        if not 1 <= stride < context:
            raise ValueError("Sliding evaluation needs 1 <= stride < context")
        previous_end = 0
        for start in range(0, len(ids), stride):
            end = min(start+context, len(ids))
            x = ids[start:end][None]
            labels = x.clone()
            if previous_end > start:
                labels[:, :previous_end-start] = -100
            labels[:, ~targets[start:end]] = -100
            count = int((labels[:, 1:] != -100).sum())
            if count:
                yield x, labels, count
            previous_end = end
            if end == len(ids):
                break
    else:
        for start in range(0, (len(ids)//context)*context, context):
            x = ids[start:start+context][None]
            labels = x.clone()
            labels[:, ~targets[start:start+context]] = -100
            count = int((labels[:, 1:] != -100).sum())
            if count:
                yield x, labels, count


@torch.inference_mode()
def calculate_ppl(model, ids, context=512, stride=256, protocol="sliding", score_mask=None):
    """Exactly reproduce the old evaluator, or use the current sliding one."""
    model.eval()
    device = next(model.parameters()).device
    total_nll, total_count = 0.0, 0
    for x, labels, count in protocol_windows(ids, context, stride, protocol, score_mask):
        output = model(input_ids=x.to(device), labels=labels.to(device),
                       use_cache=False, return_dict=True)
        loss = float(output.loss)
        if not math.isfinite(loss):
            raise ArithmeticError("Non-finite PPL loss")
        total_nll += loss*count
        total_count += count
    expected = target_mask(ids.numel(), context, protocol)
    if score_mask is not None:
        expected &= score_mask.cpu().bool()
    if total_count != int(expected.sum()):
        raise AssertionError("Incorrect next-token denominator")
    nll = total_nll/total_count
    return {"PPL": math.exp(nll) if nll < 700 else float("inf"),
            "NLL": nll, "Predicted_Tokens": total_count}


def _write_json(path, value):
    def normalize(x):
        if isinstance(x, dict):
            return {k: normalize(v) for k, v in x.items()}
        if isinstance(x, (list, tuple)):
            return [normalize(v) for v in x]
        if isinstance(x, np.generic):
            return normalize(x.item())
        if isinstance(x, float) and not math.isfinite(x):
            return None
        return x
    Path(path).write_text(json.dumps(normalize(value), ensure_ascii=False, indent=2,
                                    allow_nan=False), encoding="utf-8")



Writing /kaggle/working/evaluation_protocol.py


In [4]:
%%writefile /kaggle/working/nonlinear_composition_multi.py
"""Greedy multi-step layer compression for GPT-2.

The experiment supports 1..max_merges compression steps. At every step, all
adjacent *current groups* are considered. This distinguishes, for example:

    merge 1-2, then merge 3-4      -> groups [1,2] and [3,4]
    merge 1-2, then merge (1-2)-3  -> group  [1,2,3]

For NonlinearMultiply, an expanded group is initialized from the already merged
left block when available, then trained against the ORIGINAL teacher's complete
span mapping. Thus a three-layer group learns approximately
    Phi_3(Phi_2(Phi_1(x))) ~= Phi_merged(x)
while preserving the sequential-merging initialization.

Methods:
- LinearMultiply: fit affine maps for every original block in the merged span
  and compose them into one affine block.
- Remove: keep the first block of the combined group and remove the rest.
- Average: replace the combined group by the equal parameter average of ALL
  original teacher blocks in that group.
- NonlinearMultiply: one GPT2Block trained with hidden reconstruction + logit KD.

Selection is greedy and uses WikiText-2 validation NLL only. Final unseen tests
(WikiText-2 test, LAMBADA, C4) are never used for training or model selection.
"""
from __future__ import annotations

import copy
import gc
import hashlib
import json
import math
import platform
from contextlib import contextmanager
from dataclasses import asdict, dataclass, replace
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from threadpoolctl import threadpool_limits

import affine_core as sm
import evaluation_protocol as ep

METHODS = ("LinearMultiply", "Remove", "Average", "NonlinearMultiply")


@dataclass(frozen=True)
class Config:
    model_id: str = "openai-community/gpt2"
    model_revision: str = "main"
    dataset_revision: str = "main"
    seed: int = 42
    mode: str = "quick"
    context_length: int = 512
    eval_stride: int = 256
    eval_tokens: int | None = 10000
    hidden_eval_tokens: int = 4096
    local_tokens: int = 4096
    kd_tokens: int = 4096
    holdout_tokens: int = 2048
    epochs: int = 10
    patience: int = 15
    grad_accumulation: int = 1
    lr_transformer: float = 3e-5
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    lambda_hidden: float = 1.0
    lambda_kl: float = 1.0
    temperature: float = 2.0
    epsilon: float = 1e-8
    affine_ridge: float = 1e-4
    kl_chunk_tokens: int = 64
    cpu_threads: int = 2
    latency_seq_len: int = 256
    latency_repeats: int = 10
    max_merges: int = 5
    output_dir: str = "/kaggle/working/layer_merging_multi_results"


def make_config(mode="quick", **kwargs):
    if mode not in ("quick", "full"):
        raise ValueError("mode must be quick or full")
    cfg = Config() if mode == "quick" else Config(
        mode="full",
        local_tokens=20480,
        kd_tokens=20480,
        holdout_tokens=4096,
        epochs=15,
        eval_tokens=None,
        hidden_eval_tokens=8192,
        latency_repeats=30,
    )
    return replace(cfg, **kwargs)


def validate_config(cfg):
    if not 1 <= cfg.eval_stride < cfg.context_length:
        raise ValueError("Use overlapping sliding evaluation")
    for key in ("local_tokens", "kd_tokens", "holdout_tokens"):
        n = getattr(cfg, key)
        if n < cfg.context_length or n % cfg.context_length:
            raise ValueError(f"{key} must be a positive multiple of context_length")
    for key in (
        "epochs", "patience", "grad_accumulation", "kl_chunk_tokens",
        "latency_seq_len", "latency_repeats", "hidden_eval_tokens",
        "cpu_threads", "max_merges",
    ):
        if getattr(cfg, key) < 1:
            raise ValueError(f"{key} must be positive")
    if cfg.max_merges > 10:
        raise ValueError("For GPT-2 small, max_merges must leave at least one block")
    if cfg.temperature <= 0 or cfg.epsilon <= 0 or cfg.affine_ridge <= 0:
        raise ValueError("temperature, epsilon, affine_ridge must be positive")
    if cfg.lambda_hidden < 0 or cfg.lambda_kl < 0 or cfg.lambda_hidden + cfg.lambda_kl == 0:
        raise ValueError("Loss weights must be nonnegative and not both zero")
    if cfg.lr_transformer <= 0 or cfg.max_grad_norm <= 0 or cfg.weight_decay < 0:
        raise ValueError("Invalid optimizer configuration")


# ---------------------------------------------------------------------------
# Compression-plan utilities
# ---------------------------------------------------------------------------

def initial_groups(n_layers):
    return tuple((i,) for i in range(n_layers))


def merge_groups(groups, boundary):
    """Merge two adjacent *current* groups.

    Example (0-based):
      ((0,), (1,2), (3,), (4,)) with boundary=1
        -> ((0,), (1,2,3), (4,))
    """
    groups = tuple(tuple(g) for g in groups)
    if not 0 <= boundary < len(groups) - 1:
        raise ValueError("Invalid current-group boundary")
    left, right = groups[boundary], groups[boundary + 1]
    if left[-1] + 1 != right[0]:
        raise ValueError("Only contiguous original-layer groups may be merged")
    combined = left + right
    return groups[:boundary] + (combined,) + groups[boundary + 2:]


def validate_groups(groups, n_layers):
    flat = [x for group in groups for x in group]
    if flat != list(range(n_layers)):
        raise ValueError(f"Invalid group partition: {groups}")
    if any(group != tuple(range(group[0], group[-1] + 1)) for group in groups):
        raise ValueError("Every group must be a contiguous original-layer span")


def compressed_groups(groups):
    return [tuple(g) for g in groups if len(g) > 1]


def plan_string(groups, one_based=False):
    chunks = []
    for group in compressed_groups(groups):
        vals = [x + 1 if one_based else x for x in group]
        chunks.append(str(vals[0]) if len(vals) == 1 else f"{vals[0]}-{vals[-1]}")
    return ";".join(chunks) if chunks else "none"


def group_layout(groups, one_based=False):
    chunks = []
    for group in groups:
        vals = [x + 1 if one_based else x for x in group]
        chunks.append(str(vals[0]) if len(vals) == 1 else "[" + ",".join(map(str, vals)) + "]")
    return " | ".join(chunks)


def transition_kind(groups, boundary):
    left, right = groups[boundary], groups[boundary + 1]
    if len(left) == 1 and len(right) == 1:
        return "new_disjoint_pair"
    if len(left) > 1 and len(right) == 1:
        return "expand_left_merged_group"
    if len(left) == 1 and len(right) > 1:
        return "prepend_to_right_merged_group"
    return "join_two_merged_groups"


# ---------------------------------------------------------------------------
# Model/block utilities
# ---------------------------------------------------------------------------

def average_blocks_many(blocks):
    """Equal parameter average over ALL blocks, not iterative 50/50 averaging."""
    if len(blocks) < 2:
        raise ValueError("Need at least two blocks to average")
    merged = copy.deepcopy(blocks[0])
    states = [b.state_dict() for b in blocks]
    keys = states[0].keys()
    if any(s.keys() != keys for s in states[1:]):
        raise ValueError("Blocks have incompatible state_dict keys")
    averaged = {}
    for key in keys:
        tensors = [s[key] for s in states]
        if any(t.shape != tensors[0].shape for t in tensors[1:]):
            raise ValueError(f"Mismatched tensor for {key}")
        first = tensors[0]
        if torch.is_floating_point(first) or torch.is_complex(first):
            acc = first.clone()
            for t in tensors[1:]:
                acc = acc + t
            averaged[key] = acc / len(tensors)
        else:
            if any(not torch.equal(first, t) for t in tensors[1:]):
                raise ValueError(f"Non-floating buffer differs for {key}")
            averaged[key] = first.clone()
    merged.load_state_dict(averaged)
    return merged


def parameter_digest(model, excluded_prefix=None):
    digest = hashlib.sha256()
    for name, p in model.named_parameters():
        if excluded_prefix and name.startswith(excluded_prefix):
            continue
        digest.update(name.encode())
        digest.update(p.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def cpu_state(block):
    return {key: value.detach().cpu().clone() for key, value in block.state_dict().items()}


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _identity_affine(model):
    d = model.config.n_embd
    return np.eye(d, dtype=np.float64), np.zeros(d, dtype=np.float64)


def build_next_student(current, teacher, groups, boundary, method, cfg, affine=None, block_state=None):
    """Apply one additional compression step to `current`.

    For NonlinearMultiply, the new block is initialized from the CURRENT left
    block. Therefore [1,2] -> [1,2,3] starts from the already learned [1,2]
    block, then learns the full original teacher span 1..3.
    """
    if method not in METHODS:
        raise ValueError("Unknown method")
    validate_groups(groups, len(teacher.transformer.h))
    new_groups = merge_groups(groups, boundary)
    span = new_groups[boundary]
    start, end = span[0], span[-1]

    student = copy.deepcopy(current).eval().requires_grad_(False)
    blocks = list(student.transformer.h)
    ref = next(student.parameters())

    if method == "LinearMultiply":
        if affine is None:
            affine = _identity_affine(student)
        merged = sm.AffineMergedBlock(*affine, device=ref.device, dtype=ref.dtype)
    elif method == "Remove":
        # Keep the first CURRENT block of the combined group.
        merged = blocks[boundary]
    elif method == "Average":
        originals = [teacher.transformer.h[i] for i in span]
        merged = average_blocks_many(originals).to(ref.device, ref.dtype)
    else:  # NonlinearMultiply
        # Sequential initialization: preserve the previously merged left group.
        merged = copy.deepcopy(blocks[boundary]).to(ref.device)

    student.transformer.h = nn.ModuleList(blocks[:boundary] + [merged] + blocks[boundary + 2:])
    student.config.n_layer = len(student.transformer.h)
    student.config.use_cache = student.generation_config.use_cache = False

    if block_state is not None:
        student.transformer.h[boundary].load_state_dict(block_state)

    trainable = method == "NonlinearMultiply" and block_state is None
    student.requires_grad_(False)
    student.transformer.h[boundary].requires_grad_(trainable)
    student.eval()

    expected = {id(p) for p in student.transformer.h[boundary].parameters()} if trainable else set()
    actual = {id(p) for p in student.parameters() if p.requires_grad}
    if actual != expected:
        raise AssertionError("Only the new NonlinearMultiply block may be trainable")
    if len(student.transformer.h) != len(current.transformer.h) - 1:
        raise AssertionError("Each compression step must remove exactly one current block")
    return student, new_groups, (start, end)


# ---------------------------------------------------------------------------
# TRAIN split and objectives
# ---------------------------------------------------------------------------

def split_train(ids, cfg):
    """Create disjoint LOCAL/KD/HOLDOUT windows from WikiText-2 TRAIN only."""
    counts = [cfg.local_tokens, cfg.kd_tokens, cfg.holdout_tokens]
    starts = np.arange(0, len(ids), cfg.context_length)
    np.random.default_rng(cfg.seed).shuffle(starts)
    needed = sum(counts) // cfg.context_length
    if len(starts) < needed:
        raise ValueError("Not enough disjoint TRAIN windows")
    chosen = starts[:needed]
    if any(int(s) + cfg.context_length > len(ids) for s in chosen):
        raise ValueError("Selected a short final TRAIN window; choose another seed or use more data")
    parts, regions, offset = {}, [], 0
    for name, count in zip(("local", "kd", "holdout"), counts):
        subset = chosen[offset:offset + count // cfg.context_length]
        parts[name] = [ids[int(s):int(s) + cfg.context_length].clone() for s in subset]
        regions.extend({"Split": name, "Start": int(s), "End": int(s) + cfg.context_length} for s in subset)
        offset += len(subset)
    if len(set(int(s) for s in chosen)) != needed:
        raise AssertionError("TRAIN windows overlap")
    return parts, pd.DataFrame(regions)


@contextmanager
def capture_teacher_span(model, start, end):
    states = {}
    handles = [
        model.transformer.h[start].register_forward_pre_hook(
            lambda m, args: states.update(h=args[0])
        ),
        model.transformer.h[end].register_forward_hook(
            lambda m, args, out: states.update(target=out[0])
        ),
    ]
    try:
        yield states
    finally:
        for handle in handles:
            handle.remove()


def hidden_loss(student_h, teacher_h, cfg):
    if student_h.shape != teacher_h.shape:
        raise AssertionError("Teacher/student hidden shape mismatch")
    raw = (student_h.float() - teacher_h.float()).square().mean()
    return raw / (teacher_h.float().square().mean() + cfg.epsilon), raw


def distillation_kl(student_logits, teacher_logits, cfg):
    if student_logits.shape != teacher_logits.shape or student_logits.shape[1] < 2:
        raise ValueError("Expected matching [B,T,V] logits, T >= 2")
    s = student_logits[:, :-1].reshape(-1, student_logits.shape[-1])
    t = teacher_logits[:, :-1].reshape_as(s).detach()

    def chunk_loss(a, b):
        logp = F.log_softmax(b.float() / cfg.temperature, dim=-1)
        logq = F.log_softmax(a.float() / cfg.temperature, dim=-1)
        return (logp.exp() * (logp - logq)).sum()

    total = s.new_zeros((), dtype=torch.float32)
    for pos in range(0, len(s), cfg.kl_chunk_tokens):
        a = s[pos:pos + cfg.kl_chunk_tokens]
        b = t[pos:pos + cfg.kl_chunk_tokens]
        part = checkpoint(chunk_loss, a, b, use_reentrant=False) if a.requires_grad else chunk_loss(a, b)
        total = total + part
    return cfg.temperature ** 2 * total / len(s)


def local_objective_span(teacher, student, boundary, span, ids, cfg):
    start, end = span
    with torch.no_grad(), capture_teacher_span(teacher, start, end) as states:
        teacher.transformer(ids, use_cache=False, return_dict=True)
    prediction = student.transformer.h[boundary](states["h"], use_cache=False)[0]
    return hidden_loss(prediction, states["target"], cfg)


def kd_objective(teacher, student, ids, cfg):
    with torch.no_grad():
        teacher_logits = teacher(ids, use_cache=False, return_dict=True).logits
    student_logits = student(ids, use_cache=False, return_dict=True).logits
    return distillation_kl(student_logits, teacher_logits, cfg)


@torch.no_grad()
def reconstruction_metrics_span(teacher, student, boundary, span, windows, cfg):
    device = next(teacher.parameters()).device
    sqerr = energy = kl_sum = 0.0
    elements = tokens = 0
    normalized = []
    start, end = span
    for window in windows:
        ids = window[None].to(device)
        with capture_teacher_span(teacher, start, end) as states:
            teacher_logits = teacher(ids, use_cache=False, return_dict=True).logits
        sh = student.transformer.h[boundary](states["h"], use_cache=False)[0]
        hn, raw = hidden_loss(sh, states["target"], cfg)
        sqerr += float(raw) * sh.numel()
        energy += float(states["target"].float().square().sum())
        elements += sh.numel()
        normalized.append(float(hn))
        student_logits = student(ids, use_cache=False, return_dict=True).logits
        count = ids.shape[1] - 1
        kl_sum += float(distillation_kl(student_logits, teacher_logits, cfg)) * count
        tokens += count
    raw = sqerr / elements
    kl = kl_sum / tokens
    return {
        "Hidden_MSE": raw,
        "Normalized_Hidden_MSE": raw / (energy / elements + cfg.epsilon),
        "Mean_Window_Normalized_Hidden_MSE": float(np.mean(normalized)),
        "Teacher_KL": kl,
        "Teacher_KL_Unscaled": kl / cfg.temperature ** 2,
        "Diagnostic_Tokens": sum(len(w) for w in windows),
        "KL_Predicted_Tokens": tokens,
    }


def train_student_span(teacher, student, boundary, span, parts, cfg, stage, history_path=None):
    block = student.transformer.h[boundary]
    frozen_before = parameter_digest(student, f"transformer.h.{boundary}.")
    optimizer = torch.optim.AdamW(
        block.parameters(), lr=cfg.lr_transformer, weight_decay=cfg.weight_decay
    )
    device = next(student.parameters()).device
    history, best_loss, best_epoch, best_state, stale = [], math.inf, 0, None, 0
    steps = max(len(parts["local"]), len(parts["kd"]))
    start, end = span

    def holdout_row(epoch, train=None):
        metric = reconstruction_metrics_span(
            teacher, student, boundary, span, parts["holdout"], cfg
        )
        h = metric["Mean_Window_Normalized_Hidden_MSE"]
        k = metric["Teacher_KL"]
        row = {
            "Stage": stage,
            "Span": f"{start}-{end}",
            "Boundary": boundary,
            "Method": "NonlinearMultiply",
            "Epoch": epoch,
            "Holdout_Hidden_Loss": h,
            "Holdout_KL": k,
            "Holdout_Total_Loss": cfg.lambda_hidden * h + cfg.lambda_kl * k,
        }
        row.update(train or {
            "Train_Hidden_Loss": None,
            "Train_KL": None,
            "Train_Total_Loss": None,
        })
        if not math.isfinite(row["Holdout_Total_Loss"]):
            raise ArithmeticError("Non-finite HOLDOUT loss")
        return row

    for epoch in range(cfg.epochs + 1):
        train = None
        if epoch:
            rng = np.random.default_rng(
                cfg.seed + stage * 100003 + start * 1009 + end * 97 + epoch
            )
            order_local = rng.permutation(len(parts["local"]))
            order_kd = rng.permutation(len(parts["kd"]))
            hsum = ksum = 0.0
            optimizer.zero_grad(set_to_none=True)
            for step in range(steps):
                group_start = (step // cfg.grad_accumulation) * cfg.grad_accumulation
                divisor = min(cfg.grad_accumulation, steps - group_start)
                local = parts["local"][order_local[step % len(order_local)]][None].to(device)
                kd = parts["kd"][order_kd[step % len(order_kd)]][None].to(device)

                hn, raw = local_objective_span(teacher, student, boundary, span, local, cfg)
                if not torch.isfinite(hn):
                    raise ArithmeticError("Non-finite hidden loss")
                hsum += float(hn.detach())
                if cfg.lambda_hidden:
                    (cfg.lambda_hidden * hn / divisor).backward()
                del hn, raw

                kl = kd_objective(teacher, student, kd, cfg)
                if not torch.isfinite(kl):
                    raise ArithmeticError("Non-finite KL")
                ksum += float(kl.detach())
                if cfg.lambda_kl:
                    (cfg.lambda_kl * kl / divisor).backward()
                del kl

                if (step + 1) % cfg.grad_accumulation == 0 or step + 1 == steps:
                    torch.nn.utils.clip_grad_norm_(
                        block.parameters(), cfg.max_grad_norm, error_if_nonfinite=True
                    )
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)

            h, k = hsum / steps, ksum / steps
            train = {
                "Train_Hidden_Loss": h,
                "Train_KL": k,
                "Train_Total_Loss": cfg.lambda_hidden * h + cfg.lambda_kl * k,
            }

        row = holdout_row(epoch, train)
        history.append(row)
        if history_path:
            pd.DataFrame(history).to_csv(history_path, index=False)
        score = row["Holdout_Total_Loss"]
        if score < best_loss:
            best_loss, best_epoch, best_state, stale = score, epoch, cpu_state(block), 0
        else:
            stale += 1
        print(
            f"    Nonlinear span={start}-{end} epoch={epoch} "
            f"holdout={score:.6f} best={best_epoch}",
            flush=True,
        )
        if epoch and stale >= cfg.patience:
            break

    block.load_state_dict(best_state)
    if parameter_digest(student, f"transformer.h.{boundary}.") != frozen_before:
        raise AssertionError("Frozen weights changed")
    return history, best_epoch


# ---------------------------------------------------------------------------
# Affine multi-layer composition
# ---------------------------------------------------------------------------

@torch.inference_mode()
def collect_span_states(model, windows, start, end):
    """Return [input_to_start, output_start, ..., output_end] as CPU arrays."""
    d = model.config.n_embd
    buckets = [[] for _ in range(end - start + 2)]

    def append(slot, tensor):
        buckets[slot].append(tensor.detach().reshape(-1, d).float().cpu().numpy())

    handles = [
        model.transformer.h[start].register_forward_pre_hook(
            lambda m, args: append(0, args[0])
        )
    ]
    for layer in range(start, end + 1):
        slot = layer - start + 1
        handles.append(
            model.transformer.h[layer].register_forward_hook(
                lambda m, args, out, slot=slot: append(slot, out[0])
            )
        )
    try:
        device = next(model.parameters()).device
        for window in windows:
            model.transformer(window[None].to(device), use_cache=False, return_dict=True)
    finally:
        for handle in handles:
            handle.remove()
    return [np.concatenate(parts, axis=0) for parts in buckets]


def fit_affine_span(teacher, windows, span, cfg):
    start, end = span
    states = collect_span_states(teacher, windows, start, end)
    d = teacher.config.n_embd
    A_total = np.eye(d, dtype=np.float64)
    b_total = np.zeros(d, dtype=np.float64)
    with threadpool_limits(limits=cfg.cpu_threads):
        for x, y in zip(states[:-1], states[1:]):
            A, b = sm.fit_affine(x, y, cfg.affine_ridge)
            b_total = b_total @ A + b
            A_total = A_total @ A
    return A_total, b_total


# ---------------------------------------------------------------------------
# Checkpoint/replay: save only the NEW merged block at each stage
# ---------------------------------------------------------------------------

def save_stage_delta(path, teacher, method, stage, groups_before, groups_after,
                     boundary, span, student, cfg, best_epoch):
    payload = {
        "format_version": 2,
        "method": method,
        "stage": stage,
        "boundary": boundary,
        "span": tuple(span),
        "groups_before": tuple(tuple(g) for g in groups_before),
        "groups_after": tuple(tuple(g) for g in groups_after),
        "model_id": cfg.model_id,
        "model_revision": getattr(teacher.config, "_commit_hash", None) or cfg.model_revision,
        "original_layers": len(teacher.transformer.h),
        "config": asdict(cfg),
        "best_epoch": best_epoch,
        "block_state": None,
    }
    if method in ("LinearMultiply", "NonlinearMultiply"):
        payload["block_state"] = cpu_state(student.transformer.h[boundary])
    torch.save(payload, path)


def replay_selected_stage(run_dir, method, stage, teacher=None, device=None):
    out = Path(run_dir)
    cfg = Config(**json.loads((out / "configuration.json").read_text(encoding="utf-8")))
    if teacher is None:
        first = torch.load(out / f"selected_{method}_stage1.pt", map_location="cpu", weights_only=True)
        teacher = sm.AutoModelForCausalLM.from_pretrained(
            first["model_id"], revision=first["model_revision"],
            attn_implementation="eager", dtype=torch.float32,
        ).to(device or ("cuda" if torch.cuda.is_available() else "cpu")).eval().requires_grad_(False)
    current = copy.deepcopy(teacher).eval().requires_grad_(False)
    current.config.use_cache = current.generation_config.use_cache = False
    groups = initial_groups(len(teacher.transformer.h))

    for s in range(1, stage + 1):
        saved = torch.load(out / f"selected_{method}_stage{s}.pt", map_location="cpu", weights_only=True)
        expected_before = tuple(tuple(g) for g in saved["groups_before"])
        if tuple(tuple(g) for g in groups) != expected_before:
            raise ValueError(f"Stage replay mismatch for {method}, stage {s}")
        boundary = int(saved["boundary"])
        if method == "LinearMultiply":
            affine = _identity_affine(current)
        else:
            affine = None
        current, new_groups, span = build_next_student(
            current, teacher, groups, boundary, method, cfg,
            affine=affine, block_state=saved.get("block_state"),
        )
        current.requires_grad_(False).eval()
        groups = new_groups
        if tuple(span) != tuple(saved["span"]):
            raise ValueError("Saved span does not match replayed span")
    return current, groups


# ---------------------------------------------------------------------------
# Evaluation and greedy search
# ---------------------------------------------------------------------------

def evaluate_ppl(model, ids, cfg):
    return ep.calculate_ppl(model, ids, cfg.context_length, cfg.eval_stride, protocol="sliding")


def _candidate_row(method, stage, groups_before, groups_after, boundary, span,
                   metric, baseline, params, best_epoch, transition):
    start, end = span
    return {
        "Method": method,
        "Compression_Steps": stage,
        "Candidate_Boundary": boundary,
        "Transition": transition,
        "Merged_Span_0based": f"{start}-{end}",
        "Merged_Span_1based": f"{start+1}-{end+1}",
        "Plan_0based": plan_string(groups_after, one_based=False),
        "Plan_1based": plan_string(groups_after, one_based=True),
        "Layout_0based": group_layout(groups_after, one_based=False),
        "Layout_1based": group_layout(groups_after, one_based=True),
        "Validation_PPL": metric["PPL"],
        "Validation_NLL": metric["NLL"],
        "Predicted_Tokens": metric["Predicted_Tokens"],
        "Original_Validation_PPL": baseline["PPL"],
        "PPL_Delta_vs_Original": metric["PPL"] - baseline["PPL"],
        "PPL_Degradation_%": 100 * (metric["PPL"] / baseline["PPL"] - 1),
        "Parameters": params,
        "Parameter_Reduction_%": 100 * (1 - params / baseline["Parameters"]),
        "Remaining_Blocks": len(groups_after),
        "Best_Epoch": best_epoch,
        "Groups_Before": repr(tuple(tuple(g) for g in groups_before)),
        "Groups_After": repr(tuple(tuple(g) for g in groups_after)),
    }


def greedy_method_search(teacher, method, parts, val_ids, diag_windows, cfg, out, baseline):
    """Greedy stage-wise search for one method.

    At each stage all adjacent CURRENT groups are candidates. Therefore stage 2
    naturally compares a disjoint new pair against expanding the stage-1 group.
    """
    current = copy.deepcopy(teacher).eval().requires_grad_(False)
    current.config.use_cache = current.generation_config.use_cache = False
    groups = initial_groups(len(teacher.transformer.h))
    all_rows, selected_rows, histories = [], [], []

    for stage in range(1, cfg.max_merges + 1):
        print(f"\n===== {method}: compression stage {stage}/{cfg.max_merges} =====", flush=True)
        best = None
        best_block_state = None
        best_affine = None
        best_groups = None
        best_span = None
        best_boundary = None

        for boundary in range(len(groups) - 1):
            candidate_groups = merge_groups(groups, boundary)
            span_tuple = candidate_groups[boundary]
            span = (span_tuple[0], span_tuple[-1])
            transition = transition_kind(groups, boundary)
            print(
                f"  candidate boundary={boundary} span={span[0]}-{span[1]} "
                f"type={transition}", flush=True
            )

            affine = fit_affine_span(teacher, parts["local"], span, cfg) if method == "LinearMultiply" else None
            sm.set_seed(cfg.seed + stage * 100003 + boundary * 1009 + span[0] * 97)
            student, new_groups, built_span = build_next_student(
                current, teacher, groups, boundary, method, cfg, affine=affine
            )
            if built_span != span:
                raise AssertionError("Span bookkeeping mismatch")

            best_epoch = None
            if method == "NonlinearMultiply":
                hist_path = out / f"history_{method}_stage{stage}_b{boundary}_span{span[0]}_{span[1]}.csv"
                hist, best_epoch = train_student_span(
                    teacher, student, boundary, span, parts, cfg, stage, hist_path
                )
                histories.extend(hist)
                pd.DataFrame(histories).to_csv(out / f"training_history_{method}.csv", index=False)

            metric = evaluate_ppl(student, val_ids, cfg)
            params = sm.parameter_count(student)
            row = _candidate_row(
                method, stage, groups, new_groups, boundary, span,
                metric, baseline, params, best_epoch, transition,
            )
            # Keep candidate search relatively cheap: selection uses validation NLL only.
            # Hidden/KL diagnostics are computed only for the selected candidate after the stage.
            all_rows.append(row)
            pd.DataFrame(all_rows).to_csv(out / f"candidates_{method}.csv", index=False)

            if best is None or metric["NLL"] < best["Validation_NLL"]:
                best = dict(row)
                best_block_state = cpu_state(student.transformer.h[boundary])
                best_affine = affine
                best_groups = new_groups
                best_span = span
                best_boundary = boundary

            print(
                f"    PPL={metric['PPL']:.4f}, degradation={row['PPL_Degradation_%']:+.2f}%",
                flush=True,
            )
            del student
            cleanup()

        if best is None:
            raise RuntimeError("No candidate was evaluated")

        # Reconstruct the selected candidate without retraining.
        if method == "LinearMultiply":
            current, check_groups, _ = build_next_student(
                current, teacher, groups, best_boundary, method, cfg,
                affine=best_affine, block_state=best_block_state,
            )
        else:
            current, check_groups, _ = build_next_student(
                current, teacher, groups, best_boundary, method, cfg,
                block_state=best_block_state,
            )
        current.requires_grad_(False).eval()
        if check_groups != best_groups:
            raise AssertionError("Selected-plan reconstruction mismatch")

        if diag_windows:
            best.update(reconstruction_metrics_span(
                teacher, current, best_boundary, best_span, diag_windows, cfg
            ))

        selected_rows.append(best)
        pd.DataFrame(selected_rows).to_csv(out / f"selected_stages_{method}.csv", index=False)
        save_stage_delta(
            out / f"selected_{method}_stage{stage}.pt",
            teacher, method, stage, groups, best_groups, best_boundary,
            best_span, current, cfg, best["Best_Epoch"],
        )
        print(
            f"  >>> SELECTED {method} stage={stage}: "
            f"plan={best['Plan_0based']} PPL={best['Validation_PPL']:.4f}",
            flush=True,
        )
        groups = best_groups

    return all_rows, selected_rows


def run_experiment(cfg=None):
    cfg = cfg or make_config()
    validate_config(cfg)
    sm.check_transformers_version()
    sm.set_seed(cfg.seed)
    torch.set_num_threads(cfg.cpu_threads)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    out = Path(cfg.output_dir) / f"multi_merge_{cfg.mode}_{stamp}"
    out.mkdir(parents=True, exist_ok=False)
    ep._write_json(out / "configuration.json", asdict(cfg))
    ep._write_json(out / "protocol.json", {
        "name": "Greedy Multi-Step Adjacent-Layer Compression",
        "methods": METHODS,
        "max_merges": cfg.max_merges,
        "selection": "At each stage, evaluate every adjacent CURRENT-group merge and select minimum WikiText-2 validation NLL.",
        "overlap_semantics": "A selected merged group remains one current block. Merging it with its neighbor expands the original span, e.g. [1,2]+[3] -> [1,2,3].",
        "nonlinear_target": "New nonlinear block is initialized from current left block, but hidden target is the ORIGINAL teacher mapping across the full combined span.",
        "average": "Equal parameter average over all ORIGINAL blocks in the combined span.",
        "remove": "Keep first block of combined group; remove the rest.",
        "test_policy": "WikiText-2 test, LAMBADA, and C4 are not used during training or validation selection.",
        "search_limitation": "Greedy stage-wise search is tractable but does not guarantee the globally best combination of k merges.",
    })

    import datasets
    dataset = datasets.load_dataset(
        "Salesforce/wikitext", "wikitext-2-raw-v1", revision=cfg.dataset_revision
    )
    tokenizer = sm.AutoTokenizer.from_pretrained(cfg.model_id, revision=cfg.model_revision)
    tokenizer.model_max_length = 10**12

    def tokenize(split, limit=None):
        text = "\n\n".join(t for t in dataset[split]["text"] if t and t.strip())
        ids = tokenizer(text, return_tensors="pt", add_special_tokens=False)["input_ids"][0]
        return ids if limit is None else ids[:limit]

    train_ids = tokenize("train")
    val_ids = tokenize("validation", cfg.eval_tokens)
    parts, regions = split_train(train_ids, cfg)
    regions.to_csv(out / "train_regions.csv", index=False)

    teacher = sm.AutoModelForCausalLM.from_pretrained(
        cfg.model_id, revision=cfg.model_revision,
        attn_implementation="eager", dtype=torch.float32,
    ).to(device).eval().requires_grad_(False)
    teacher.config.use_cache = teacher.generation_config.use_cache = False
    layers = len(teacher.transformer.h)
    if cfg.max_merges >= layers:
        raise ValueError("max_merges must be smaller than the number of transformer blocks")
    if cfg.model_id == "openai-community/gpt2" and layers != 12:
        raise AssertionError("Expected 12 GPT-2 blocks")
    if max(cfg.context_length, cfg.latency_seq_len) > teacher.config.n_positions:
        raise ValueError("Sequence length exceeds GPT-2 context")

    teacher_digest = parameter_digest(teacher)
    ep._write_json(out / "environment.json", {
        "python": platform.python_version(),
        "torch": str(torch.__version__),
        "transformers": sm.transformers.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": sm.scipy.__version__,
        "datasets": datasets.__version__,
        "device": str(device),
        "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
        "model_resolved_revision": getattr(teacher.config, "_commit_hash", None) or cfg.model_revision,
        "dataset_requested_revision": cfg.dataset_revision,
        "dataset_fingerprints": {name: part._fingerprint for name, part in dataset.items()},
        "teacher_parameter_sha256": teacher_digest,
    })

    baseline = {
        **evaluate_ppl(teacher, val_ids, cfg),
        "Parameters": sm.parameter_count(teacher),
    }
    ep._write_json(out / "baseline_validation.json", baseline)
    diag_windows = sm.make_state_windows(
        val_ids, min(cfg.hidden_eval_tokens, len(val_ids)), cfg.context_length
    )
    diag_windows = [w for w in diag_windows if len(w) >= 2]

    print(f"Device: {device}; output: {out}", flush=True)
    print(f"Original validation: {baseline}", flush=True)

    candidate_rows, selected_rows = [], []
    for method in METHODS:
        rows, selected = greedy_method_search(
            teacher, method, parts, val_ids, diag_windows, cfg, out, baseline
        )
        candidate_rows.extend(rows)
        selected_rows.extend(selected)
        pd.DataFrame(candidate_rows).to_csv(out / "all_candidates.csv", index=False)
        pd.DataFrame(selected_rows).to_csv(out / "selected_by_stage.csv", index=False)

    if parameter_digest(teacher) != teacher_digest:
        raise AssertionError("Original teacher changed")

    selected_frame = pd.DataFrame(selected_rows)
    selected_frame = selected_frame.sort_values(["Compression_Steps", "Method"]).reset_index(drop=True)
    selected_frame.to_csv(out / "selected_by_stage.csv", index=False)

    pivot = selected_frame.pivot(
        index="Compression_Steps", columns="Method", values="Validation_PPL"
    ).reindex(columns=METHODS)
    pivot.to_csv(out / "validation_ppl_by_compression.csv")

    checkpoint_names = [
        f"selected_{method}_stage{stage}.pt"
        for method in METHODS for stage in range(1, cfg.max_merges + 1)
    ]
    lock_files = ["configuration.json", "selected_by_stage.csv", *checkpoint_names]
    ep._write_json(out / "selection_locked.json", {
        "basis": "Greedy per-stage WikiText-2 VALIDATION NLL; nonlinear epoch selected on TRAIN HOLDOUT only.",
        "max_merges": cfg.max_merges,
        "file_sha256": {
            name: hashlib.sha256((out / name).read_bytes()).hexdigest()
            for name in lock_files
        },
    })

    del teacher
    cleanup()
    return {
        "output_dir": str(out),
        "candidates": pd.DataFrame(candidate_rows),
        "selected": selected_frame,
        "pivot": pivot,
        "baseline": baseline,
    }


# ---------------------------------------------------------------------------
# FINAL unseen-data evaluation. Never used for selection.
# ---------------------------------------------------------------------------

def _load_text_ids(tokenizer, source_name, eval_tokens, cfg):
    import datasets
    if source_name == "WikiText2-test":
        ds = datasets.load_dataset(
            "Salesforce/wikitext", "wikitext-2-raw-v1", revision=cfg.dataset_revision,
            split="test",
        )
        texts = (row["text"] for row in ds)
    elif source_name == "LAMBADA":
        ds = datasets.load_dataset("EleutherAI/lambada_openai", split="test")
        texts = (row["text"] for row in ds)
    elif source_name == "C4":
        ds = datasets.load_dataset("allenai/c4", "en", split="validation", streaming=True)
        texts = (row["text"] for row in ds)
    else:
        raise ValueError(f"Unknown unseen dataset: {source_name}")

    pieces, total = [], 0
    sep = tokenizer("\n\n", add_special_tokens=False)["input_ids"]
    for text in texts:
        if text is None or not text.strip():
            continue
        if total and sep:
            t = torch.tensor(sep, dtype=torch.long)
            pieces.append(t)
            total += t.numel()
        ids = tokenizer(text, return_tensors="pt", add_special_tokens=False)["input_ids"][0].cpu()
        pieces.append(ids)
        total += ids.numel()
        if eval_tokens is not None and total >= eval_tokens:
            break
    if not pieces:
        raise RuntimeError(f"No usable text in {source_name}")
    ids = torch.cat(pieces)
    return ids if eval_tokens is None else ids[:eval_tokens]


def run_unseen_test(run_dir, stages=(2, 3, 4, 5), datasets_to_test=("WikiText2-test", "LAMBADA", "C4"),
                    eval_tokens=10000, force=False):
    """Evaluate locked stage checkpoints on data never used for training/selection."""
    out = Path(run_dir)
    result_path = out / "unseen_test_results.csv"
    if result_path.exists() and not force:
        raise RuntimeError("Unseen test already exists. Use force=True only if you intentionally want to rerun it.")

    locked = json.loads((out / "selection_locked.json").read_text(encoding="utf-8"))
    for name, expected in locked["file_sha256"].items():
        if hashlib.sha256((out / name).read_bytes()).hexdigest() != expected:
            raise ValueError("Locked selection/config/checkpoint changed: " + name)

    cfg = Config(**json.loads((out / "configuration.json").read_text(encoding="utf-8")))
    env = json.loads((out / "environment.json").read_text(encoding="utf-8"))
    stages = tuple(int(s) for s in stages)
    if any(s < 1 or s > cfg.max_merges for s in stages):
        raise ValueError("Requested test stage is outside 1..max_merges")

    sm.check_transformers_version()
    sm.set_seed(cfg.seed)
    torch.set_num_threads(cfg.cpu_threads)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = sm.AutoTokenizer.from_pretrained(
        cfg.model_id, revision=env["model_resolved_revision"]
    )
    tokenizer.model_max_length = 10**12
    teacher = sm.AutoModelForCausalLM.from_pretrained(
        cfg.model_id, revision=env["model_resolved_revision"],
        attn_implementation="eager", dtype=torch.float32,
    ).to(device).eval().requires_grad_(False)
    teacher.config.use_cache = teacher.generation_config.use_cache = False
    if parameter_digest(teacher) != env["teacher_parameter_sha256"]:
        raise ValueError("Original weights differ from validation run")

    rows = []
    for dataset_name in datasets_to_test:
        print("\n" + "=" * 72)
        print(f"UNSEEN DATASET: {dataset_name}")
        print("=" * 72)
        ids = _load_text_ids(tokenizer, dataset_name, eval_tokens, cfg)
        baseline = evaluate_ppl(teacher, ids, cfg)
        print(f"Tokens={ids.numel():,}; Original PPL={baseline['PPL']:.4f}")
        rows.append({
            "Dataset": dataset_name,
            "Method": "Original",
            "Compression_Steps": 0,
            "Remaining_Blocks": len(teacher.transformer.h),
            "Plan_0based": "none",
            "Plan_1based": "none",
            **baseline,
            "PPL_Delta_vs_Original": 0.0,
            "PPL_Degradation_%": 0.0,
            "NLL_Delta_vs_Original": 0.0,
        })

        for stage in stages:
            for method in METHODS:
                student, groups = replay_selected_stage(out, method, stage, teacher=teacher)
                metric = evaluate_ppl(student, ids, cfg)
                row = {
                    "Dataset": dataset_name,
                    "Method": method,
                    "Compression_Steps": stage,
                    "Remaining_Blocks": len(groups),
                    "Plan_0based": plan_string(groups, False),
                    "Plan_1based": plan_string(groups, True),
                    **metric,
                    "PPL_Delta_vs_Original": metric["PPL"] - baseline["PPL"],
                    "PPL_Degradation_%": 100 * (metric["PPL"] / baseline["PPL"] - 1),
                    "NLL_Delta_vs_Original": metric["NLL"] - baseline["NLL"],
                }
                rows.append(row)
                print(
                    f"stage={stage} {method:19s} plan={row['Plan_0based']:16s} "
                    f"PPL={metric['PPL']:.4f} degradation={row['PPL_Degradation_%']:+.2f}%"
                )
                del student
                cleanup()
        pd.DataFrame(rows).to_csv(result_path, index=False)

    frame = pd.DataFrame(rows)
    frame.to_csv(result_path, index=False)
    del teacher
    cleanup()
    return frame

Writing /kaggle/working/nonlinear_composition_multi.py


In [5]:
%%writefile /kaggle/working/verify_multi.py
"""Small offline-style structural checks for the multi-step notebook.
Runs on a randomly initialized tiny GPT-2; no pretrained-quality claims.
"""
import copy
import nonlinear_composition_multi as nc
import torch
from transformers import GPT2Config, GPT2LMHeadModel


def run_checks():
    nc.sm.check_transformers_version()
    nc.sm.set_seed(123)
    checks = []

    def passed(name):
        checks.append({"check": name, "status": "PASS"})

    # Plan semantics: disjoint and chained stage-2 merges are different.
    g0 = nc.initial_groups(6)
    g1 = nc.merge_groups(g0, 1)                  # [1,2]
    chain = nc.merge_groups(g1, 1)               # [1,2,3]
    disjoint = nc.merge_groups(g1, 2)            # [1,2] and [3,4]
    assert chain != disjoint
    assert nc.plan_string(chain) == "1-3"
    assert nc.plan_string(disjoint) == "1-2;3-4"
    passed("disjoint_vs_chained_plan_semantics")

    tiny = GPT2LMHeadModel(GPT2Config(
        vocab_size=101, n_positions=32, n_ctx=32, n_embd=32,
        n_layer=6, n_head=4, n_inner=64,
        resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0,
    )).eval().requires_grad_(False)
    tiny.config.use_cache = tiny.generation_config.use_cache = False
    cfg = nc.make_config(
        mode="quick", model_id="synthetic-test-only", context_length=8,
        eval_stride=4, local_tokens=8, kd_tokens=8, holdout_tokens=8,
        hidden_eval_tokens=8, max_merges=5, epochs=1,
    )

    # Remove: every step removes exactly one current block.
    cur = copy.deepcopy(tiny)
    groups = nc.initial_groups(6)
    cur, groups, _ = nc.build_next_student(cur, tiny, groups, 1, "Remove", cfg)
    cur, groups, _ = nc.build_next_student(cur, tiny, groups, 1, "Remove", cfg)
    assert len(cur.transformer.h) == 4
    assert nc.plan_string(groups) == "1-3"
    passed("multi_step_remove")

    # Average [1,2,3] is an equal 1/3 average of ORIGINAL teacher blocks.
    cur = copy.deepcopy(tiny)
    groups = nc.initial_groups(6)
    cur, groups, _ = nc.build_next_student(cur, tiny, groups, 1, "Average", cfg)
    cur, groups, _ = nc.build_next_student(cur, tiny, groups, 1, "Average", cfg)
    expected = nc.average_blocks_many([tiny.transformer.h[i] for i in (1, 2, 3)])
    for a, b in zip(cur.transformer.h[1].parameters(), expected.parameters()):
        assert torch.allclose(a, b)
    passed("uniform_multi_layer_average")

    # Nonlinear chained initialization uses the current merged left block.
    cur = copy.deepcopy(tiny)
    groups = nc.initial_groups(6)
    cur, groups, _ = nc.build_next_student(cur, tiny, groups, 1, "NonlinearMultiply", cfg)
    with torch.no_grad():
        first_param = next(cur.transformer.h[1].parameters())
        first_param.add_(0.123)
    cur.requires_grad_(False)
    before = {k: v.detach().clone() for k, v in cur.transformer.h[1].state_dict().items()}
    cur2, groups2, span = nc.build_next_student(cur, tiny, groups, 1, "NonlinearMultiply", cfg)
    after = cur2.transformer.h[1].state_dict()
    assert span == (1, 3)
    assert all(torch.equal(before[k], after[k]) for k in before)
    passed("nonlinear_chain_initializes_from_previous_merge")

    # Only the newly created nonlinear block is trainable.
    trainable_ids = {id(p) for p in cur2.parameters() if p.requires_grad}
    expected_ids = {id(p) for p in cur2.transformer.h[1].parameters()}
    assert trainable_ids == expected_ids
    passed("only_new_nonlinear_block_trainable")

    return checks


if __name__ == "__main__":
    import json
    print(json.dumps(run_checks(), indent=2))


Writing /kaggle/working/verify_multi.py


## 3. Self-Checks

Using a small randomly initialized GPT-2, we verify at least the following:

- `1-2;3-4` (non-overlapping) and `1-3` (chained) are represented as different plans
- Applying Remove repeatedly reduces the number of blocks by one per step
- A three-layer Average becomes an **equal-weight average of 1/3 each** over the three original blocks
- In a chained nonlinear merge, the previously merged left block is used to initialize the next merge
- Only the newly created nonlinear merged block is trainable

Passing these checks verifies the code structure; it does not guarantee the quality of the pretrained GPT-2 model.


In [6]:
import sys, importlib, json
from pathlib import Path

sys.path.insert(0, '/kaggle/working')
import affine_core
import evaluation_protocol
import nonlinear_composition_multi as nc
import verify_multi

importlib.reload(affine_core)
importlib.reload(evaluation_protocol)
importlib.reload(nc)
importlib.reload(verify_multi)

checks = verify_multi.run_checks()
print(json.dumps(checks, indent=2))
Path('/kaggle/working/multi_self_checks.json').write_text(
    json.dumps(checks, indent=2), encoding='utf-8'
)
assert all(x['status'] == 'PASS' for x in checks)


[
  {
    "check": "disjoint_vs_chained_plan_semantics",
    "status": "PASS"
  },
  {
    "check": "multi_step_remove",
    "status": "PASS"
  },
  {
    "check": "uniform_multi_layer_average",
    "status": "PASS"
  },
  {
    "check": "nonlinear_chain_initializes_from_previous_merge",
    "status": "PASS"
  },
  {
    "check": "only_new_nonlinear_block_trainable",
    "status": "PASS"
  }
]


## 4. Experimental Setup

- We sequentially explore `Compression_Steps=1..5`.
- One step = combining two currently adjacent groups into one group = reducing the number of Transformer blocks by one.
- At stage 2, for example, both `1-2;3-4` and `1-3` correspond to "two compression operations."
- Each method searches stage 2 starting from **its own best stage-1 model**, and continues in the same way through stage 5.
- Only WikiText-2 validation NLL is used for selection.
- For NonlinearMultiply, epoch selection uses only the disjoint HOLDOUT subset within TRAIN.
- The unseen test sets (WikiText-2 test / LAMBADA / C4) are never used for selection at any point.

### Chained Nonlinear Merge

For example, if `[1,2]` has already been trained and is then merged with `3`, we:

1. Initialize the new block from the **current `[1,2]` merged block**
2. Feed it the input to the teacher's original layer 1
3. Use the output of the teacher's original layer 3 as the target

This way, the previous merge result is reused while accumulated error is corrected against the original teacher's full-span target.


In [7]:
import json
import torch
import nonlinear_composition_multi as nc

# Start with 'quick' to verify the structure, then use 'full' for the main experiment
MODE = 'full'
MAX_MERGES = 5

CFG = nc.make_config(
    MODE,
    max_merges=MAX_MERGES,
    # In full mode, the default is 15 epochs
    # epochs=15,
    # patience=15,
    # grad_accumulation=2,
)

nc.validate_config(CFG)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print(json.dumps(nc.asdict(CFG), indent=2))

if not torch.cuda.is_available():
    raise RuntimeError('Please set Accelerator to GPU in Kaggle Settings.')


GPU available: True
GPU: Tesla T4
{
  "model_id": "openai-community/gpt2",
  "model_revision": "main",
  "dataset_revision": "main",
  "seed": 42,
  "mode": "full",
  "context_length": 512,
  "eval_stride": 256,
  "eval_tokens": null,
  "hidden_eval_tokens": 8192,
  "local_tokens": 20480,
  "kd_tokens": 20480,
  "holdout_tokens": 4096,
  "epochs": 15,
  "patience": 15,
  "grad_accumulation": 1,
  "lr_transformer": 3e-05,
  "weight_decay": 0.01,
  "max_grad_norm": 1.0,
  "lambda_hidden": 1.0,
  "lambda_kl": 1.0,
  "temperature": 2.0,
  "epsilon": 1e-08,
  "affine_ridge": 0.0001,
  "kl_chunk_tokens": 64,
  "cpu_threads": 2,
  "latency_seq_len": 256,
  "latency_repeats": 30,
  "max_merges": 5,
  "output_dir": "/kaggle/working/layer_merging_multi_results"
}


## 5. Multi-step Training / Validation Search

For each method, stages 1→5 are explored sequentially.

For example, if `[1,2]` is selected at stage 1, then at stage 2 all boundaries in the current group sequence are tested, so at least the following candidates are evaluated separately:

- Merge `[1,2]` with `3` to form `[1,2,3]`
- Merge a different pair, `3` and `4`, to form `[1,2]` + `[3,4]`

**Note:** Because this is a greedy search, stage-2 plans that begin from candidates discarded at stage 1 are not explored. This is much cheaper than exhaustively searching all `C(11,k)` plans, but it does not guarantee the global optimum.


In [8]:
results = nc.run_experiment(CFG)
print('Saved to:', results['output_dir'])


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Device: cuda; output: /kaggle/working/layer_merging_multi_results/multi_merge_full_20260930T184306_417427Z
Original validation: {'PPL': 28.611473516724676, 'NLL': 3.35380780929132, 'Predicted_Tokens': 249748, 'Parameters': 124439808}

===== LinearMultiply: compression stage 1/5 =====
  candidate boundary=0 span=0-1 type=new_disjoint_pair
    PPL=63.4196, degradation=+121.66%
  candidate boundary=1 span=1-2 type=new_disjoint_pair
    PPL=38.4182, degradation=+34.28%
  candidate boundary=2 span=2-3 type=new_disjoint_pair
    PPL=40.5943, degradation=+41.88%
  candidate boundary=3 span=3-4 type=new_disjoint_pair
    PPL=40.3671, degradation=+41.09%
  candidate boundary=4 span=4-5 type=new_disjoint_pair
    PPL=37.0539, degradation=+29.51%
  candidate boundary=5 span=5-6 type=new_disjoint_pair
    PPL=37.1711, degradation=+29.92%
  candidate boundary=6 span=6-7 type=new_disjoint_pair
    PPL=36.3555, degradation=+27.07%
  candidate boundary=7 span=7-8 type=new_disjoint_pair
    PPL=36.8708

## 6. Validation Results (Especially for 2–5 Compression Steps)

`Plan_0based` / `Plan_1based` make it possible to distinguish non-overlapping merges from chained merges.

Examples:
- `1-2;3-4` = two independent merged groups
- `1-3` = one three-layer merged group (two merge operations)

`Transition` also records whether the current stage created a new independent pair or expanded an existing merged group.


In [9]:
from pathlib import Path
import pandas as pd

selected = results['selected'].copy()
show = selected[selected['Compression_Steps'].between(2, 5)]
columns = [
    'Compression_Steps', 'Method', 'Transition',
    'Merged_Span_0based', 'Plan_0based', 'Plan_1based',
    'Validation_PPL', 'PPL_Degradation_%',
    'Remaining_Blocks', 'Parameters', 'Parameter_Reduction_%', 'Best_Epoch'
]
display(show[columns])

print('===== Validation PPL by compression steps =====')
display(results['pivot'].loc[2:5])

print('===== Nonlinear selected path =====')
display(
    selected[(selected['Method'] == 'NonlinearMultiply') & selected['Compression_Steps'].between(1, 5)][columns]
)

print('Saved:', results['output_dir'])


,Compression_Steps,Method,Transition,Merged_Span_0based,Plan_0based,Plan_1based,Validation_PPL,PPL_Degradation_%,Remaining_Blocks,Parameters,Parameter_Reduction_%,Best_Epoch
4,2,Average,prepend_to_right_merged_group,6-8,6-8,7-9,76.028504,165.727329,10,110264064,11.391647,NaN
5,2,LinearMultiply,expand_left_merged_group,6-8,6-8,7-9,44.790395,56.546969,10,103766784,16.612870,NaN
6,2,NonlinearMultiply,new_disjoint_pair,3-4,0-1;3-4,1-2;4-5,31.687210,10.750012,10,110264064,11.391647,15.0
7,2,Remove,new_disjoint_pair,8-9,0-1;8-9,1-2;9-10,35.124442,22.763484,10,110264064,11.391647,NaN
8,3,Average,prepend_to_right_merged_group,5-8,5-8,6-9,138.955558,385.663760,9,103176192,17.087471,NaN
9,3,LinearMultiply,expand_left_merged_group,6-9,6-9,7-10,60.751634,112.333120,9,96678912,22.308694,NaN
10,3,NonlinearMultiply,new_disjoint_pair,7-8,0-1;3-4;7-8,1-2;4-5;8-9,35.250592,23.204392,9,103176192,17.087471,15.0
11,3,Remove,new_disjoint_pair,2-3,0-1;2-3;8-9,1-2;3-4;9-10,43.067961,50.526887,9,103176192,17.087471,NaN
12,4,Average,prepend_to_right_merged_group,4-8,4-8,5-9,238.948228,735.148278,8,96088320,22.783295,NaN
13,4,LinearMultiply,new_disjoint_pair,1-2,1-2;6-9,2-3;7-10,87.215960,204.828619,8,83093760,33.225741,NaN


===== Validation PPL by compression steps =====


Method,LinearMultiply,Remove,Average,NonlinearMultiply
Compression_Steps,,,,
2,44.790395,35.124442,76.028504,31.687210
3,60.751634,43.067961,138.955558,35.250592
4,87.215960,58.026434,238.948228,40.958538
5,122.052678,89.100453,424.979200,48.579084


===== Nonlinear selected path =====


,Compression_Steps,Method,Transition,Merged_Span_0based,Plan_0based,Plan_1based,Validation_PPL,PPL_Degradation_%,Remaining_Blocks,Parameters,Parameter_Reduction_%,Best_Epoch
2,1,NonlinearMultiply,new_disjoint_pair,0-1,0-1,1-2,29.402309,2.764049,11,117351936,5.695824,15.0
6,2,NonlinearMultiply,new_disjoint_pair,3-4,0-1;3-4,1-2;4-5,31.687210,10.750012,10,110264064,11.391647,15.0
10,3,NonlinearMultiply,new_disjoint_pair,7-8,0-1;3-4;7-8,1-2;4-5;8-9,35.250592,23.204392,9,103176192,17.087471,15.0
14,4,NonlinearMultiply,new_disjoint_pair,5-6,0-1;3-4;5-6;7-8,1-2;4-5;6-7;8-9,40.958538,43.154241,8,96088320,22.783295,15.0
18,5,NonlinearMultiply,new_disjoint_pair,9-10,0-1;3-4;5-6;7-8;9-10,1-2;4-5;6-7;8-9;10-11,48.579084,69.788824,7,89000448,28.479118,15.0


Saved: /kaggle/working/layer_merging_multi_results/multi_merge_full_20260930T184306_417427Z


## 7. How to Interpret the Results

The following three points are especially important in this experiment.

1. **Does the gap between Remove and NonlinearMultiply widen as the number of compression steps increases?**
   - Even if Remove performs well with one compression step, we can check whether its PPL deteriorates sharply after 3–5 steps.
2. **Is the selected plan non-overlapping or chained?**
   - This can be checked using `Plan` and `Transition`.
3. **Is the external generalization of NonlinearMerge maintained?**
   - The stage 2–5 checkpoints selected on validation data are evaluated directly on unseen data.

For Average, even in chained merges, we do not use iterative 50/50 averaging. Instead, all original blocks in the combined span are averaged with equal weights. For example, three layers receive weights `1/3,1/3,1/3`. This avoids weight bias caused by the merge order.


## 8. Unseen TEST — Data Not Used for Training/Validation

After the selection is completely locked, only the models from stages **2,3,4,5** are evaluated on the following datasets:

- `WikiText2-test`: held-out test data from the same corpus
- `LAMBADA`: external data with a book / narrative style
- `C4`: external web-text data

The PPL values on these datasets are **never used for pair/group selection or epoch selection**. When interpreting the results, compare `PPL_Degradation_%` relative to the Original GPT-2 within each dataset rather than comparing raw PPL values across datasets.


In [10]:
RUN_UNSEEN_TEST = True
UNSEEN_EVAL_TOKENS = 10_000

if RUN_UNSEEN_TEST:
    unseen_results = nc.run_unseen_test(
        results['output_dir'],
        stages=(2, 3, 4, 5),
        datasets_to_test=('WikiText2-test', 'LAMBADA', 'C4'),
        eval_tokens=UNSEEN_EVAL_TOKENS,
    )
    display(unseen_results)
else:
    print('Unseen test is disabled.')



UNSEEN DATASET: WikiText2-test
Tokens=10,000; Original PPL=33.0396
stage=2 LinearMultiply      plan=6-8              PPL=53.5152 degradation=+61.97%
stage=2 Remove              plan=0-1;8-9          PPL=40.7905 degradation=+23.46%
stage=2 Average             plan=6-8              PPL=90.6788 degradation=+174.45%
stage=2 NonlinearMultiply   plan=0-1;3-4          PPL=37.0165 degradation=+12.04%
stage=3 LinearMultiply      plan=6-9              PPL=73.5531 degradation=+122.62%
stage=3 Remove              plan=0-1;2-3;8-9      PPL=50.8156 degradation=+53.80%
stage=3 Average             plan=5-8              PPL=163.8812 degradation=+396.01%
stage=3 NonlinearMultiply   plan=0-1;3-4;7-8      PPL=40.9416 degradation=+23.92%
stage=4 LinearMultiply      plan=1-2;6-9          PPL=108.0617 degradation=+227.07%
stage=4 Remove              plan=0-1;2-3;7-9      PPL=69.4953 degradation=+110.34%
stage=4 Average             plan=4-8              PPL=272.1427 degradation=+723.69%
stage=4 NonlinearMult

README.md: 0.00B [00:00, ?B/s]

default/test/default.parquet:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/5153 [00:00<?, ? examples/s]

Tokens=10,000; Original PPL=42.9060
stage=2 LinearMultiply      plan=6-8              PPL=80.3697 degradation=+87.32%
stage=2 Remove              plan=0-1;8-9          PPL=47.1597 degradation=+9.91%
stage=2 Average             plan=6-8              PPL=78.2065 degradation=+82.27%
stage=2 NonlinearMultiply   plan=0-1;3-4          PPL=49.1481 degradation=+14.55%
stage=3 LinearMultiply      plan=6-9              PPL=130.3904 degradation=+203.90%
stage=3 Remove              plan=0-1;2-3;8-9      PPL=57.5907 degradation=+34.23%
stage=3 Average             plan=5-8              PPL=118.5571 degradation=+176.32%
stage=3 NonlinearMultiply   plan=0-1;3-4;7-8      PPL=53.8207 degradation=+25.44%
stage=4 LinearMultiply      plan=1-2;6-9          PPL=227.1660 degradation=+429.45%
stage=4 Remove              plan=0-1;2-3;7-9      PPL=68.8842 degradation=+60.55%
stage=4 Average             plan=4-8              PPL=169.4623 degradation=+294.96%
stage=4 NonlinearMultiply   plan=0-1;3-4;5-6;7-8  PPL=5

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Tokens=10,000; Original PPL=33.2230
stage=2 LinearMultiply      plan=6-8              PPL=61.5437 degradation=+85.24%
stage=2 Remove              plan=0-1;8-9          PPL=38.3852 degradation=+15.54%
stage=2 Average             plan=6-8              PPL=77.0010 degradation=+131.77%
stage=2 NonlinearMultiply   plan=0-1;3-4          PPL=37.6100 degradation=+13.20%
stage=3 LinearMultiply      plan=6-9              PPL=94.9550 degradation=+185.81%
stage=3 Remove              plan=0-1;2-3;8-9      PPL=46.1668 degradation=+38.96%
stage=3 Average             plan=5-8              PPL=119.0389 degradation=+258.30%
stage=3 NonlinearMultiply   plan=0-1;3-4;7-8      PPL=41.9357 degradation=+26.22%
stage=4 LinearMultiply      plan=1-2;6-9          PPL=147.7780 degradation=+344.81%
stage=4 Remove              plan=0-1;2-3;7-9      PPL=58.0528 degradation=+74.74%
stage=4 Average             plan=4-8              PPL=172.0414 degradation=+417.84%
stage=4 NonlinearMultiply   plan=0-1;3-4;5-6;7-8  PPL=

,Dataset,Method,Compression_Steps,Remaining_Blocks,Plan_0based,Plan_1based,PPL,NLL,Predicted_Tokens,PPL_Delta_vs_Original,PPL_Degradation_%,NLL_Delta_vs_Original
0,WikiText2-test,Original,0,12,none,none,33.039640,3.497708,9999,0.000000,0.000000,0.000000
1,WikiText2-test,LinearMultiply,2,10,6-8,7-9,53.515218,3.979966,9999,20.475578,61.972764,0.482258
2,WikiText2-test,Remove,2,10,0-1;8-9,1-2;9-10,40.790493,3.708449,9999,7.750853,23.459255,0.210741
3,WikiText2-test,Average,2,10,6-8,7-9,90.678784,4.507323,9999,57.639144,174.454515,1.009615
4,WikiText2-test,NonlinearMultiply,2,10,0-1;3-4,1-2;4-5,37.016514,3.611364,9999,3.976874,12.036676,0.113656
5,WikiText2-test,LinearMultiply,3,9,6-9,7-10,73.553077,4.298007,9999,40.513437,122.620697,0.800299
6,WikiText2-test,Remove,3,9,0-1;2-3;8-9,1-2;3-4;9-10,50.815580,3.928203,9999,17.775940,53.801856,0.430495
7,WikiText2-test,Average,3,9,5-8,6-9,163.881209,5.099142,9999,130.841569,396.013906,1.601434
8,WikiText2-test,NonlinearMultiply,3,9,0-1;3-4;7-8,1-2;4-5;8-9,40.941614,3.712147,9999,7.901974,23.916648,0.214439
9,WikiText2-test,LinearMultiply,4,8,1-2;6-9,2-3;7-10,108.061663,4.682702,9999,75.022023,227.066710,1.184994


In [11]:
if RUN_UNSEEN_TEST:
    compressed = unseen_results[unseen_results['Method'] != 'Original'].copy()

    print('===== PPL degradation (%) =====')
    degradation = compressed.pivot_table(
        index=['Dataset', 'Compression_Steps'],
        columns='Method',
        values='PPL_Degradation_%'
    )
    display(degradation)

    print('===== Selected plans used on unseen data =====')
    display(
        compressed[['Dataset', 'Compression_Steps', 'Method', 'Plan_0based',
                    'PPL', 'PPL_Degradation_%']]
        .sort_values(['Dataset', 'Compression_Steps', 'Method'])
    )


===== PPL degradation (%) =====


Method                                Average  LinearMultiply  \
Dataset        Compression_Steps                                
C4             2                   131.770012       85.244123   
               3                   258.302662      185.810964   
               4                   417.838164      344.806048   
               5                   688.661565      568.882228   
LAMBADA        2                    82.274119       87.315879   
               3                   176.318526      203.898234   
               4                   294.962220      429.451090   
               5                   458.945819      683.227796   
WikiText2-test 2                   174.454515       61.972764   
               3                   396.013906      122.620697   
               4                   723.685296      227.066710   
               5                  1338.937776      367.504645   

Method                            NonlinearMultiply      Remove  
Dataset        Compression_Steps                                 
C4             2                          13.204513   15.538014  
               3                          26.224786   38.960211  
               4                          48.122299   74.736661  
               5                          74.934264  126.140978  
LAMBADA        2                          14.548449    9.914183  
               3                          25.438848   34.225440  
               4                          38.281264   60.547009  
               5                          59.342436   98.533648  
WikiText2-test 2                          12.036676   23.459255  
               3                          23.916648   53.801856  
               4                          44.858334  110.339225  
               5                          72.100929  221.170913

===== Selected plans used on unseen data =====


,Dataset,Compression_Steps,Method,Plan_0based,PPL,PPL_Degradation_%
37,C4,2,Average,6-8,77.000983,131.770012
35,C4,2,LinearMultiply,6-8,61.543681,85.244123
38,C4,2,NonlinearMultiply,0-1;3-4,37.609951,13.204513
36,C4,2,Remove,0-1;8-9,38.385210,15.538014
41,C4,3,Average,5-8,119.038943,258.302662
39,C4,3,LinearMultiply,6-9,94.955016,185.810964
42,C4,3,NonlinearMultiply,0-1;3-4;7-8,41.935678,26.224786
40,C4,3,Remove,0-1;2-3;8-9,46.166770,38.960211
45,C4,4,Average,4-8,172.041445,417.838164
43,C4,4,LinearMultiply,1-2;6-9,147.777975,344.806048


## 9. Saving, ZIP Packaging, and Reloading

Instead of saving the entire full GPT-2 model at every stage, each selected checkpoint stores only the **newly created merged block at that stage** as a delta.

- Remove / Average: no block state needs to be saved because the model can be deterministically reconstructed from the teacher
- LinearMultiply / NonlinearMultiply: only the state of the newly merged block is saved

When loading a stage-4 model, the deltas from stages 1→4 are replayed sequentially. This approach keeps checkpoint size manageable even across multiple stages.


In [12]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

run_dir = Path(results['output_dir'])
for name in [
    'affine_core.py', 'evaluation_protocol.py',
    'nonlinear_composition_multi.py', 'verify_multi.py', 'multi_self_checks.json'
]:
    src = Path('/kaggle/working') / name
    if src.exists():
        shutil.copy2(src, run_dir / name)

archive = shutil.make_archive(str(run_dir), 'zip', root_dir=run_dir)
print('ZIP saved to:', archive)
display(FileLink(archive))


ZIP saved to: /kaggle/working/layer_merging_multi_results/multi_merge_full_20260930T184306_417427Z.zip


/kaggle/working/layer_merging_multi_results/multi_merge_full_20260930T184306_417427Z.zip

In [13]:
# Example: reconstruct the selected NonlinearMultiply model at stage 3
# Do not reselect using test/validation data; replay the locked delta checkpoints instead.
def load_selected_nonlinear_stage(run_dir, stage=3, teacher=None):
    model, groups = nc.replay_selected_stage(
        run_dir,
        method='NonlinearMultiply',
        stage=stage,
        teacher=teacher,
    )
    return model, groups

# model, groups = load_selected_nonlinear_stage(results['output_dir'], stage=3)
# print('plan:', nc.plan_string(groups))


## References / Experimental Notes

- Hinton et al., *Distilling the Knowledge in a Neural Network*
- Transformers 4.57.6 GPT-2 implementation
- Sliding PPL / affine fitting protocol from the original notebook

### Important Definitions in This Implementation

- **Compression step**: An operation that combines two currently adjacent groups into one group. Each step reduces the number of blocks by one.
- **Chained nonlinear merge**: Initialize from the merged left block from the previous stage, then retrain against the combined full-span target from the original teacher.
- **Average**: Take an equal-weight average of the original blocks in the combined span. This is not iterative 50/50 averaging.
- **Greedy limitation**: At each stage, the search proceeds only from the best model of the previous stage, rather than exhaustively searching all plans.
- **Unseen test**: WikiText-2 test / LAMBADA / C4 are not used for training or validation selection.

`Plan_0based` matches the Python/GPT-2 block indices. `Plan_1based` is a human-readable one-based representation.
